In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("PJME_hourly.csv")
df.head()

,Datetime,PJME_MW
0,2002-12-31 01:00:00,26498.0
1,2002-12-31 02:00:00,25147.0
2,2002-12-31 03:00:00,24574.0
3,2002-12-31 04:00:00,24393.0
4,2002-12-31 05:00:00,24860.0


In [3]:
df["Datetime"] = pd.to_datetime(df["Datetime"])

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 145366 entries, 0 to 145365
Data columns (total 2 columns):
 #   Column    Non-Null Count   Dtype         
---  ------    --------------   -----         
 0   Datetime  145366 non-null  datetime64[us]
 1   PJME_MW   145366 non-null  float64       
dtypes: datetime64[us](1), float64(1)
memory usage: 2.2 MB


In [5]:
df = df.sort_values("Datetime").reset_index(drop=True)

In [6]:
print(
    "Chronologically sorted:",
    df["Datetime"].is_monotonic_increasing
)

Chronologically sorted: True


In [7]:
df["Datetime"].duplicated().sum()

np.int64(4)

In [8]:
duplicates = df[
    df["Datetime"].duplicated(keep=False)
].sort_values("Datetime")

duplicates.head(20)

,Datetime,PJME_MW
112487,2014-11-02 02:00:00,23755.0
112488,2014-11-02 02:00:00,22935.0
121223,2015-11-01 02:00:00,21567.0
121224,2015-11-01 02:00:00,21171.0
130127,2016-11-06 02:00:00,20795.0
130128,2016-11-06 02:00:00,21692.0
138863,2017-11-05 02:00:00,21236.0
138864,2017-11-05 02:00:00,20666.0


In [9]:
df = (
    df.groupby("Datetime", as_index=False)["PJME_MW"]
    .mean()
)

In [11]:
df["Datetime"].duplicated().sum()

np.int64(0)

In [12]:
df = df.set_index("Datetime")

In [13]:
df.head()

,PJME_MW
Datetime,
2002-01-01 01:00:00,30393.0
2002-01-01 02:00:00,29265.0
2002-01-01 03:00:00,28357.0
2002-01-01 04:00:00,27899.0
2002-01-01 05:00:00,28057.0


In [14]:
df.index

DatetimeIndex(['2002-01-01 01:00:00', '2002-01-01 02:00:00',
               '2002-01-01 03:00:00', '2002-01-01 04:00:00',
               '2002-01-01 05:00:00', '2002-01-01 06:00:00',
               '2002-01-01 07:00:00', '2002-01-01 08:00:00',
               '2002-01-01 09:00:00', '2002-01-01 10:00:00',
               ...
               '2018-08-02 15:00:00', '2018-08-02 16:00:00',
               '2018-08-02 17:00:00', '2018-08-02 18:00:00',
               '2018-08-02 19:00:00', '2018-08-02 20:00:00',
               '2018-08-02 21:00:00', '2018-08-02 22:00:00',
               '2018-08-02 23:00:00', '2018-08-03 00:00:00'],
              dtype='datetime64[us]', name='Datetime', length=145362, freq=None)

In [15]:
print(
    df.index.to_series().diff().value_counts().head()
)

Datetime
0 days 01:00:00    145331
0 days 02:00:00        30
Name: count, dtype: int64


In [16]:
full_index = pd.date_range(
    start=df.index.min(),
    end=df.index.max(),
    freq="h"
)

In [17]:
print(
    "Expected hourly records:",
    len(full_index)
)
print(
    "Existing records:",
    len(df)
)

Expected hourly records: 145392
Existing records: 145362


In [18]:
df = df.reindex(full_index)

In [19]:
df.index.name = "Datetime"

In [20]:
df.head()

,PJME_MW
Datetime,
2002-01-01 01:00:00,30393.0
2002-01-01 02:00:00,29265.0
2002-01-01 03:00:00,28357.0
2002-01-01 04:00:00,27899.0
2002-01-01 05:00:00,28057.0


In [21]:
print(df.isnull().sum())

PJME_MW    30
dtype: int64


In [22]:
missing_demand = df["PJME_MW"].isnull().sum()
print(
    "Missing demand values:",
    missing_demand
)

Missing demand values: 30


In [23]:
missing_percentage = (
    df["PJME_MW"].isnull().mean() * 100
)
print(
    "Missing demand percentage:",
    missing_percentage
)

Missing demand percentage: 0.020633872565203038


In [24]:
missing_dates = df[
    df["PJME_MW"].isnull()
].index
print(missing_dates[:20])

DatetimeIndex(['2002-04-07 03:00:00', '2002-10-27 02:00:00',
               '2003-04-06 03:00:00', '2003-10-26 02:00:00',
               '2004-04-04 03:00:00', '2004-10-31 02:00:00',
               '2005-04-03 03:00:00', '2005-10-30 02:00:00',
               '2006-04-02 03:00:00', '2006-10-29 02:00:00',
               '2007-03-11 03:00:00', '2007-11-04 02:00:00',
               '2008-03-09 03:00:00', '2008-11-02 02:00:00',
               '2009-03-08 03:00:00', '2009-11-01 02:00:00',
               '2010-03-14 03:00:00', '2010-11-07 02:00:00',
               '2010-12-10 00:00:00', '2011-03-13 03:00:00'],
              dtype='datetime64[us]', name='Datetime', freq=None)


In [25]:
df["PJME_MW"] = df["PJME_MW"].interpolate(
    method="time"
)

In [26]:
print(
    "Remaining missing values:",
    df["PJME_MW"].isnull().sum()
)

Remaining missing values: 0


In [27]:
print(
    df["PJME_MW"].isnull().sum()
)

0


In [28]:
df["PJME_MW"] = df["PJME_MW"].ffill()
df["PJME_MW"] = df["PJME_MW"].bfill()

In [29]:
print(
    "Remaining missing values:",
    df["PJME_MW"].isnull().sum()
)

Remaining missing values: 0


In [30]:
negative_values = (
    df["PJME_MW"] < 0
).sum()
print(
    "Negative demand values:",
    negative_values
)

Negative demand values: 0


In [31]:
zero_demand = (
    df["PJME_MW"] == 0
).sum()
print(
    "Zero demand values:",
    zero_demand
)

Zero demand values: 0


In [32]:
Q1 = df["PJME_MW"].quantile(0.25)
Q3 = df["PJME_MW"].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)

Lower bound: 15457.0
Upper bound: 47761.0


In [33]:
outliers = df[
    (df["PJME_MW"] < lower_bound) |
    (df["PJME_MW"] > upper_bound)
]
print(
    "Potential outliers:",
    len(outliers)
)

Potential outliers: 3460


In [34]:
df["Hour"] = df.index.hour

In [35]:
df["Day"] = df.index.day

In [36]:
df["DayOfWeek"] = df.index.dayofweek

In [37]:
df["Month"] = df.index.month

In [38]:
df["Year"] = df.index.year

In [39]:
df["Week"] = df.index.isocalendar().week.astype(int)

In [40]:
df.head()

,PJME_MW,Hour,Day,DayOfWeek,Month,Year,Week
Datetime,,,,,,,
2002-01-01 01:00:00,30393.0,1,1,1,1,2002,1
2002-01-01 02:00:00,29265.0,2,1,1,1,2002,1
2002-01-01 03:00:00,28357.0,3,1,1,1,2002,1
2002-01-01 04:00:00,27899.0,4,1,1,1,2002,1
2002-01-01 05:00:00,28057.0,5,1,1,1,2002,1


In [41]:
df["IsWeekend"] = (df["DayOfWeek"] >= 5).astype(int)

In [42]:
df[["DayOfWeek", "IsWeekend"]].head(10)

,DayOfWeek,IsWeekend
Datetime,,
2002-01-01 01:00:00,1,0
2002-01-01 02:00:00,1,0
2002-01-01 03:00:00,1,0
2002-01-01 04:00:00,1,0
2002-01-01 05:00:00,1,0
2002-01-01 06:00:00,1,0
2002-01-01 07:00:00,1,0
2002-01-01 08:00:00,1,0
2002-01-01 09:00:00,1,0


In [43]:
def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    else:
        return "Autumn"

In [44]:
df["Season"] = df["Month"].apply(get_season)

In [45]:
df["Season"].value_counts()

Season
Spring    37536
Summer    36841
Winter    36071
Autumn    34944
Name: count, dtype: int64

In [46]:
df["Hour_sin"] = np.sin(2 * np.pi * df["Hour"] / 24)
df["Hour_cos"] = np.cos(2 * np.pi * df["Hour"] / 24)

In [47]:
df["DayOfWeek_sin"] = np.sin(2 * np.pi * df["DayOfWeek"] / 7)
df["DayOfWeek_cos"] = np.cos(2 * np.pi * df["DayOfWeek"] / 7)

In [48]:
df["Month_sin"] = np.sin(2 * np.pi * df["Month"] / 12)
df["Month_cos"] = np.cos(2 * np.pi * df["Month"] / 12)

In [49]:
df["Lag_1"] = df["PJME_MW"].shift(1)

In [50]:
df["Lag_24"] = df["PJME_MW"].shift(24)

In [51]:
df["Lag_48"] = df["PJME_MW"].shift(48)

In [52]:
df["Lag_168"] = df["PJME_MW"].shift(168)

In [53]:
df["RollingMean_24"] = (df["PJME_MW"].rolling(window=24).mean())

In [54]:
df["RollingMean_168"] = (df["PJME_MW"].rolling(window=168).mean())

In [55]:
df["RollingStd_24"] = (df["PJME_MW"].rolling(window=24).std())

In [56]:
df["RollingStd_168"] = (df["PJME_MW"].rolling(window=168).std())

In [57]:
df.isnull().sum()

PJME_MW              0
Hour                 0
Day                  0
DayOfWeek            0
Month                0
Year                 0
Week                 0
IsWeekend            0
Season               0
Hour_sin             0
Hour_cos             0
DayOfWeek_sin        0
DayOfWeek_cos        0
Month_sin            0
Month_cos            0
Lag_1                1
Lag_24              24
Lag_48              48
Lag_168            168
RollingMean_24      23
RollingMean_168    167
RollingStd_24       23
RollingStd_168     167
dtype: int64

In [58]:
df = df.dropna()

In [61]:
print(df.isnull().sum())

PJME_MW            0
Hour               0
Day                0
DayOfWeek          0
Month              0
Year               0
Week               0
IsWeekend          0
Season             0
Hour_sin           0
Hour_cos           0
DayOfWeek_sin      0
DayOfWeek_cos      0
Month_sin          0
Month_cos          0
Lag_1              0
Lag_24             0
Lag_48             0
Lag_168            0
RollingMean_24     0
RollingMean_168    0
RollingStd_24      0
RollingStd_168     0
dtype: int64


In [62]:
print("Final shape:", df.shape)
df.head()

Final shape: (145224, 23)


,PJME_MW,Hour,Day,DayOfWeek,Month,Year,Week,IsWeekend,Season,Hour_sin,...,Month_sin,Month_cos,Lag_1,Lag_24,Lag_48,Lag_168,RollingMean_24,RollingMean_168,RollingStd_24,RollingStd_168
Datetime,,,,,,,,,,,,,,,,,,,,,
2002-01-08 01:00:00,29445.0,1,8,1,1,2002,2,0,Winter,0.258819,...,0.5,0.866025,31187.0,26862.0,27100.0,30393.0,33560.208333,32513.869048,4425.965952,3861.770954
2002-01-08 02:00:00,28670.0,2,8,1,1,2002,2,0,Winter,0.500000,...,0.5,0.866025,29445.0,25976.0,26097.0,29265.0,33672.458333,32510.327381,4256.159403,3865.039821
2002-01-08 03:00:00,28375.0,3,8,1,1,2002,2,0,Winter,0.707107,...,0.5,0.866025,28670.0,25641.0,25793.0,28357.0,33786.375000,32510.434524,4064.104959,3864.924245
2002-01-08 04:00:00,28542.0,4,8,1,1,2002,2,0,Winter,0.866025,...,0.5,0.866025,28375.0,25666.0,25657.0,27899.0,33906.208333,32514.261905,3851.076461,3860.646270
2002-01-08 05:00:00,29261.0,5,8,1,1,2002,2,0,Winter,0.965926,...,0.5,0.866025,28542.0,26328.0,25778.0,28057.0,34028.416667,32521.428571,3640.941409,3853.433314


In [63]:
df.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 145224 entries, 2002-01-08 01:00:00 to 2018-08-03 00:00:00
Freq: h
Data columns (total 23 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   PJME_MW          145224 non-null  float64
 1   Hour             145224 non-null  int32  
 2   Day              145224 non-null  int32  
 3   DayOfWeek        145224 non-null  int32  
 4   Month            145224 non-null  int32  
 5   Year             145224 non-null  int32  
 6   Week             145224 non-null  int64  
 7   IsWeekend        145224 non-null  int64  
 8   Season           145224 non-null  str    
 9   Hour_sin         145224 non-null  float64
 10  Hour_cos         145224 non-null  float64
 11  DayOfWeek_sin    145224 non-null  float64
 12  DayOfWeek_cos    145224 non-null  float64
 13  Month_sin        145224 non-null  float64
 14  Month_cos        145224 non-null  float64
 15  Lag_1            145224 non-null  float64
 16  Lag_24 

In [64]:
print(df.columns.tolist())

['PJME_MW', 'Hour', 'Day', 'DayOfWeek', 'Month', 'Year', 'Week', 'IsWeekend', 'Season', 'Hour_sin', 'Hour_cos', 'DayOfWeek_sin', 'DayOfWeek_cos', 'Month_sin', 'Month_cos', 'Lag_1', 'Lag_24', 'Lag_48', 'Lag_168', 'RollingMean_24', 'RollingMean_168', 'RollingStd_24', 'RollingStd_168']


In [65]:
df.to_csv("PJME_preprocessed.csv")
print("Processed dataset saved successfully.")

Processed dataset saved successfully.
